In [3]:
%config Completer.use_jedi = False
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.express as px

df_raw=pd.read_csv('000_df_RAW.csv', low_memory=False)

df_raw

,Num_Acc,place,catu,grav,sexe,trajet,secu,locp,actp,etatp,...,vosp,prof,plan,lartpc,larrout,surf,infra,situ,env1,vma
0,200500000001,1.0,1,4,1,1.0,11.0,0.0,0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
1,200500000001,1.0,1,3,2,3.0,11.0,0.0,0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
2,200500000001,2.0,2,1,1,0.0,11.0,0.0,0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
3,200500000001,4.0,2,1,1,0.0,31.0,0.0,0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
4,200500000001,5.0,2,1,1,0.0,11.0,0.0,0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,1.0,1,4,1,0.0,NaN,0.0,0,-1.0,...,0.0,2.0,2.0,NaN,-1,2.0,0.0,1.0,NaN,80.0
2509722,202100056516,1.0,1,4,1,5.0,NaN,0.0,0,-1.0,...,0.0,2.0,2.0,NaN,-1,2.0,0.0,1.0,NaN,80.0
2509723,202100056517,1.0,1,3,1,0.0,NaN,0.0,0,-1.0,...,0.0,1.0,2.0,NaN,-1,2.0,5.0,3.0,NaN,80.0
2509724,202100056518,1.0,1,3,1,3.0,NaN,-1.0,-1,-1.0,...,0.0,1.0,1.0,NaN,-1,2.0,0.0,1.0,NaN,70.0


In [12]:
#******************************************On rend toutes les infos bizarres en NaN********************************************************
df=df_raw.copy(deep=True)
print(df.isna().sum().sum())

Nan_originaux=df.isna().sum()

#Variables de usagers___________________________________________________________________________________________________________________________________
df.place.replace(0,np.NaN,inplace=True) #0 est non défini pour place
df.catu.replace(4,np.NaN,inplace=True) #4 est non défini pour catu
df.grav.replace(-1,np.NaN,inplace=True) #-1 est non défini pour grav
df.sexe.replace(-1,np.NaN,inplace=True) #-1 est non défini pour sexe
df.trajet.replace([-1,0],np.NaN,inplace=True) #-1 et 0 sont non défini
#secu sera géré plus tard
df.locp.replace(-1,0,inplace=True) #-1 est non renseigné et 0 est sans objet, on rassemble les deux
df.loc[df['catu'] == 3, 'locp'] = df.loc[df['catu'] == 3, 'locp'].replace(0, np.NaN)#Localisation du piéton filtrée pour les cas où catu==3 (concerne les piétons) : on NaN la ligne si non resneignée

df['actp']=df.actp.replace({'0.0':0,'-1':0,' -1':0,'0':0,'B':0,'A' : 99 }).astype(float)#-1 est non renseigné et 0 est sans objet,B est inconnu, on rassemble tout. astype float permet de mélanger les formats x et x.0
df.loc[df['catu'] == 3, 'actp'] = df.loc[df['catu'] == 3, 'actp'].replace(0, np.NaN)#Localisation du piéton filtrée pour les cas où catu==3 (concerne les piétons) : on NaN la ligne si non resneignée

df['etatp']=df.etatp.replace({-1:0})#-1 est non renseigné et 0 est sans objet, on assemble
df.loc[df['catu'] == 3, 'etatp'] = df.loc[df['catu'] == 3, 'etatp'].replace(0, np.NaN)#Etat seul ou accompagné du piéton filtrée pour les cas où catu==3 (concerne les piétons) : NaN la ligne si non resneignée
#an_nais sera gérée plus tard
#id_vehicule à supprimer
#secu1/2/3 aussi

#Variables de vehi___________________________________________________________________________________________________________________________________
df.senc.replace([-1,0,3],np.NaN,inplace=True) #-1 est non défini pour place, 0 =inconnu et 3 est Absence d'info
df.catv.replace([-1,0],np.NaN,inplace=True) #-1 est non défini pour place, 0 =indéterminable
#occutc ne concerne que les transports en communs. On s'en occupera plus tard si vraiment on veut s'en servir, mais je pense qu'on va juste retirer la colonne.
#obs et obsm se completent surement, on vérifiera plus tard
df.choc.replace(-1,np.NaN,inplace=True) #-1 = non renseigné. On garde 0 car il représente "aucun choc", qui peut arriver dans certains accident moteurs par exemple
df.manv.replace([-1,0],np.NaN,inplace=True) #-1 Non renseigné. 0 = Inconnu
#id vehicule sert dans al clé de merge puis est supprimée
#motor n'est présent que depuis 2019, pas assez de donnée, on supprimera la colonne, si on veut travailler seulement sur 2019-2021 dans un second temps, on recommencera le préprocessing de toutes façons.



#Variables de lieux___________________________________________________________________________________________________________________________________

#Ok pour catr
#voie sera supprimée car elle a 32000 modalités différentes et est sans interet
#v1 sera supprimée car elle a 50% de nan , et la quasi totalité de ce qu'il reste est 0 ou -1 et est sans intéret
#v2 sera supprimée, un savant mélange des problèmes de voie et v1
df.circ.replace([-1,0],np.NaN,inplace=True) # -1 Non renseigné, 0 n'a pas d'explication dans les documents, et pas de sens puisque les autres modalités possibles couvrent tous les cas
df.nbv.replace([-1,0],np.NaN,inplace=True) # -1 = non resneigné je suppose. Est-ce que 0 a du sens pour un nombre de voies ? Y a-t-il des accidents dans des lieux sans voies ? Parking ? Même un parking possède des sens de circulation. Pas clair, on retire.
df.vosp.replace(-1,np.NaN,inplace=True) # -1 = non resneigné 
#pr représente la borne kilométrique la plus proche. inutile puisqu'on ne peut pas utiliser les noms de voies
#pr1 représente la distance à la borne la plus proche, inutile puisqu'on ne peut pas utiliser les noms de voies
df.prof.replace([-1,0],np.NaN,inplace=True) # Inclinaison de la route : -1 = non resneigné, 0 n'est pas expliqué dans les documents. On met NaN, mais je pense que le fillna pourra prendre 1=Plat
df.plan.replace([-1,0],np.NaN,inplace=True) # courbure de la route : -1 = non resneigné, 0 n'est pas expliqué dans les documents. On met NaN, mais je pense que le fillna pourra prendre 1=rectiligne
#lartpc : la ligne lieux.lartpc.str.replace(',','.').astype(float).describe() permet de voir la répartition de la variable : 75% valent 0. Je pense qu'on supprimera la colonne.
#larrout : largeur de la route possède une trop grosse part de 0 et -1. A supprimer je pense
df.surf.replace([-1,0],np.NaN,inplace=True)#Etat de surface : 0 et -1 sont non renseigné
df.infra.replace(-1,np.NaN,inplace=True) #Amenagement : -1 non renseigné. On garde les 0 = Aucun
df.situ.replace([-1,0],np.NaN,inplace=True) #Situation de l'accident : -1 non resneigné, 0 = aucun (n'a pas de sens)
#env1 : 15% de Nan, 3 modalités, dont la moitié pour 0. Je propose de jeter si on ne sait pas ce que c'est. (aucune explication dans les documents)
#vma n'existe que depuis 2019



#Variables de carac___________________________________________________________________________________________________________________________________

#an ok
#mois ok
#jour ok
#hrmn ok
df.lum.replace(-1,np.NaN,inplace=True) #-1 est non défini
#agg ok
df.int.replace([-1,0],np.NaN,inplace=True) #-1 et 0 sont non défini
df.atm.replace(-1,np.NaN,inplace=True) #-1 est non défini
df.col.replace(-1,np.NaN,inplace=True) #-1 est non renseigné
#adr sera supprimée car inexploitable : 467491 modalités différentes
#gps sera traité plus tard
#lat sera traité plus tard
#long sera traité plus tard
#dep peut être abandonné ou traité plus tard



Nan_réels=df.isna().sum()

df_temp1=df
print(df.isna().sum().sum())
df

25092019
28810124


,Num_Acc,place,catu,grav,sexe,trajet,secu,locp,actp,etatp,...,vosp,prof,plan,lartpc,larrout,surf,infra,situ,env1,vma
0,200500000001,1.0,1.0,4.0,1.0,1.0,11.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
1,200500000001,1.0,1.0,3.0,2.0,3.0,11.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
2,200500000001,2.0,2.0,1.0,1.0,NaN,11.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
3,200500000001,4.0,2.0,1.0,1.0,NaN,31.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
4,200500000001,5.0,2.0,1.0,1.0,NaN,11.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0,63,1.0,0.0,1.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,1.0,1.0,4.0,1.0,NaN,NaN,0.0,0.0,0.0,...,0.0,2.0,2.0,NaN,-1,2.0,0.0,1.0,NaN,80.0
2509722,202100056516,1.0,1.0,4.0,1.0,5.0,NaN,0.0,0.0,0.0,...,0.0,2.0,2.0,NaN,-1,2.0,0.0,1.0,NaN,80.0
2509723,202100056517,1.0,1.0,3.0,1.0,NaN,NaN,0.0,0.0,0.0,...,0.0,1.0,2.0,NaN,-1,2.0,5.0,3.0,NaN,80.0
2509724,202100056518,1.0,1.0,3.0,1.0,3.0,NaN,0.0,0.0,0.0,...,0.0,1.0,1.0,NaN,-1,2.0,0.0,1.0,NaN,70.0


In [48]:
#**************************************************On retire les colonnes avec trop de NaN et aucune solution pour les remplacer, où les colonnes inutilisables************************************************
df=df_temp1.copy(deep=True)
print(df.isna().sum().sum())

#infos venant de 'vehicule' : ces colonnes n'existent qu'a partir de 2019
df=df.drop(columns=['id_vehicule','motor','occutc'])

#infos venant de 'lieux' : 
df=df.drop(columns=['voie','v1','v2','lartpc','larrout','env1','vma'])

#Variables de carac_____________________________
df=df.drop(columns=['adr'])



df_temp2=df
print(df.isna().sum().sum())
df

28810124
16514922


,Num_Acc,place,catu,grav,sexe,trajet,secu,locp,actp,etatp,...,circ,nbv,pr,pr1,vosp,prof,plan,surf,infra,situ
0,200500000001,1.0,1.0,4.0,1.0,1.0,11.0,0.0,0.0,0.0,...,2.0,2.0,1.0,430.0,0.0,1.0,1.0,1.0,0.0,1.0
1,200500000001,1.0,1.0,3.0,2.0,3.0,11.0,0.0,0.0,0.0,...,2.0,2.0,1.0,430.0,0.0,1.0,1.0,1.0,0.0,1.0
2,200500000001,2.0,2.0,1.0,1.0,NaN,11.0,0.0,0.0,0.0,...,2.0,2.0,1.0,430.0,0.0,1.0,1.0,1.0,0.0,1.0
3,200500000001,4.0,2.0,1.0,1.0,NaN,31.0,0.0,0.0,0.0,...,2.0,2.0,1.0,430.0,0.0,1.0,1.0,1.0,0.0,1.0
4,200500000001,5.0,2.0,1.0,1.0,NaN,11.0,0.0,0.0,0.0,...,2.0,2.0,1.0,430.0,0.0,1.0,1.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,1.0,1.0,4.0,1.0,NaN,NaN,0.0,0.0,0.0,...,2.0,2.0,49,24,0.0,2.0,2.0,2.0,0.0,1.0
2509722,202100056516,1.0,1.0,4.0,1.0,5.0,NaN,0.0,0.0,0.0,...,2.0,2.0,49,24,0.0,2.0,2.0,2.0,0.0,1.0
2509723,202100056517,1.0,1.0,3.0,1.0,NaN,NaN,0.0,0.0,0.0,...,2.0,2.0,(1),(1),0.0,1.0,2.0,2.0,5.0,3.0
2509724,202100056518,1.0,1.0,3.0,1.0,3.0,NaN,0.0,0.0,0.0,...,2.0,2.0,2,290,0.0,1.0,1.0,2.0,0.0,1.0


In [54]:
#Melange des infos sur secu et secu1/2/3 puis dummification pour retirer les colonnes secu
df=df_temp2.copy(deep=True)
print(df.isna().sum().sum())


#uniformisation des données sur "secu" avec leur encodage sur 'secu1'
temp=df.secu.copy(deep=True)
temp[temp<=10]=0
temp[temp%10==3]=0
temp[temp%10==2]=0
temp[temp%10==0]=0
temp=temp//10


df.secu1=df.secu1.replace(np.NaN,0)
temp=temp.replace(np.NaN,0)

df["secu1"]=df.secu1+temp

dummies=df[['secu1','secu2','secu3']]

#On ne garde que certains éléments, les autres sont extremement minoritaires, entre autre car jusqu'a 2018, seuls ces 4 là sont renseignés :
dummies=dummies.replace({1:'Ceinture',2:'Casque',3:'Dispositif_Enfant',4:'Equipement_Réfléchissant',5:np.NaN,6:np.NaN,7:np.NaN,0:np.NaN,8:np.NaN,-1:np.NaN,9:np.NaN})

dummies = pd.get_dummies(dummies)

df['Casque']=dummies.secu1_Casque+dummies.secu2_Casque+dummies.secu3_Casque
df['Ceinture']=dummies.secu1_Ceinture+dummies.secu2_Ceinture+dummies.secu3_Ceinture
df['Dispositif_Enfant']=dummies.secu1_Dispositif_Enfant+dummies.secu2_Dispositif_Enfant+dummies.secu3_Dispositif_Enfant
df['Equipement_Réfléchissant']=dummies.secu1_Equipement_Réfléchissant+dummies.secu2_Equipement_Réfléchissant+dummies.secu3_Equipement_Réfléchissant

df=df.drop(columns=['secu','secu1','secu2']) #on retire les colonnes devenues inutiles
df_temp3=df
print(df.isna().sum().sum())
df

16514922
11806486


,Num_Acc,place,catu,grav,sexe,trajet,locp,actp,etatp,an_nais,...,vosp,prof,plan,surf,infra,situ,Casque,Ceinture,Dispositif_Enfant,Equipement_Réfléchissant
0,200500000001,1.0,1.0,4.0,1.0,1.0,0.0,0.0,0.0,1976.0,...,0.0,1.0,1.0,1.0,0.0,1.0,False,True,False,False
1,200500000001,1.0,1.0,3.0,2.0,3.0,0.0,0.0,0.0,1968.0,...,0.0,1.0,1.0,1.0,0.0,1.0,False,True,False,False
2,200500000001,2.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,1964.0,...,0.0,1.0,1.0,1.0,0.0,1.0,False,True,False,False
3,200500000001,4.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,2004.0,...,0.0,1.0,1.0,1.0,0.0,1.0,False,False,True,False
4,200500000001,5.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,1998.0,...,0.0,1.0,1.0,1.0,0.0,1.0,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,1.0,1.0,4.0,1.0,NaN,0.0,0.0,0.0,1975.0,...,0.0,2.0,2.0,2.0,0.0,1.0,False,True,False,False
2509722,202100056516,1.0,1.0,4.0,1.0,5.0,0.0,0.0,0.0,2002.0,...,0.0,2.0,2.0,2.0,0.0,1.0,False,True,False,False
2509723,202100056517,1.0,1.0,3.0,1.0,NaN,0.0,0.0,0.0,1999.0,...,0.0,1.0,2.0,2.0,5.0,3.0,False,False,False,False
2509724,202100056518,1.0,1.0,3.0,1.0,3.0,0.0,0.0,0.0,1968.0,...,0.0,1.0,1.0,2.0,0.0,1.0,False,True,False,False


In [79]:
df=df_temp3.copy(deep=True)
print(df.isna().sum().sum())

#date column in the datetime format
df.an.replace({5:2005,6:2006,7:2007,8:2008,9:2009},inplace=True) #fix an format
df.an.replace({10:2010,11:2011,12:2012,13:2013,14:2014,15:2015,16:2016,17:2017,18:2018},inplace=True) #fix an format
df['date'] = df['an'].astype(str) + '-' + df['mois'].astype(str) + '-' + df['jour'].astype(str)
df['date'] = pd.to_datetime(df['date'], yearfirst = True)
df['hrmn2']=df['hrmn'].astype(str) #fix hrmn format
df['hrmn2']=df['hrmn2'].apply(lambda x: x.replace(':', '') if ':' in x else x) #fix hrmn format
df['hrmn2'] = df['hrmn2'].apply(lambda x: str(x).zfill(4))  # corrige les heures à minuit dont les 0 de gauche n'ont pas été rentrés
df['hrmn2'] = pd.to_datetime(df['hrmn2'], format='%H%M', errors='raise')  #fix hrmn format
df['date'] = df['date'] + pd.to_timedelta(df['hrmn2'].dt.hour, unit='h') + pd.to_timedelta(df['hrmn2'].dt.minute, unit='m') #adding hour+time
df=df.drop(['hrmn2','hrmn','an','mois','jour'],axis=1) #removing dummy columns


df_temp4=df
print(df.isna().sum().sum())
df

11806486
11806486


,Num_Acc,place,catu,grav,sexe,trajet,locp,actp,etatp,an_nais,...,prof,plan,surf,infra,situ,Casque,Ceinture,Dispositif_Enfant,Equipement_Réfléchissant,date
0,200500000001,1.0,1.0,4.0,1.0,1.0,0.0,0.0,0.0,1976.0,...,1.0,1.0,1.0,0.0,1.0,False,True,False,False,2005-01-12 19:00:00
1,200500000001,1.0,1.0,3.0,2.0,3.0,0.0,0.0,0.0,1968.0,...,1.0,1.0,1.0,0.0,1.0,False,True,False,False,2005-01-12 19:00:00
2,200500000001,2.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,1964.0,...,1.0,1.0,1.0,0.0,1.0,False,True,False,False,2005-01-12 19:00:00
3,200500000001,4.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,2004.0,...,1.0,1.0,1.0,0.0,1.0,False,False,True,False,2005-01-12 19:00:00
4,200500000001,5.0,2.0,1.0,1.0,NaN,0.0,0.0,0.0,1998.0,...,1.0,1.0,1.0,0.0,1.0,False,True,False,False,2005-01-12 19:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509721,202100056516,1.0,1.0,4.0,1.0,NaN,0.0,0.0,0.0,1975.0,...,2.0,2.0,2.0,0.0,1.0,False,True,False,False,2021-01-01 18:00:00
2509722,202100056516,1.0,1.0,4.0,1.0,5.0,0.0,0.0,0.0,2002.0,...,2.0,2.0,2.0,0.0,1.0,False,True,False,False,2021-01-01 18:00:00
2509723,202100056517,1.0,1.0,3.0,1.0,NaN,0.0,0.0,0.0,1999.0,...,1.0,2.0,2.0,5.0,3.0,False,False,False,False,2021-01-01 10:55:00
2509724,202100056518,1.0,1.0,3.0,1.0,3.0,0.0,0.0,0.0,1968.0,...,1.0,1.0,2.0,0.0,1.0,False,True,False,False,2021-01-02 18:00:00
